# Test-set evaluation of the selected classification schemes

Each scheme (metaheuristic + objective function + classifier) is trained on the training data and evaluated
once on a test set. The 70-gene model (all genes) is evaluated the same way.

**Run all cells from the top.** Everything you may want to change is in section 1: choose the test set with
`TEST_NAME` (`'test'` for the 30-patient hold-out, `'NKI33'` for NKI33). The results files are named after it.

| Section | What it does |
|---|---|
| 1. Setup and dataset configuration | imports, and every file the notebook reads or writes |
| 2. Functions | data, metrics, classifiers, evaluation |
| 3. Dataset | loads the training and test data, scales the test data, checks both |
| 4. Evaluation | the 15 schemes and the 70-gene model, written to the results files |
| 5. Results | shows the results and writes them to Excel |

The LaTeX table is built from the hold-out results (`test_results_*.csv`) with
`python latex_tables/performance_test_table.py`.

## 1. Setup and dataset configuration

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import ast
from sklearn import metrics
import pandas as pd
import numpy as np
import os

# All paths in this notebook are relative to the repository root, the folder that holds this notebook.
# VS Code gives the notebook's own path in __vsc_ipynb_file__, so this works wherever the drive is mounted.
# Plain Jupyter already starts the kernel in the notebook's folder.
if '__vsc_ipynb_file__' in globals():
    os.chdir(os.path.dirname(__vsc_ipynb_file__))
assert os.path.isdir('data'), f'Start the kernel in the repository root: the working directory is {os.getcwd()}'

In [3]:
# ---- Dataset: every file this notebook reads or writes (paths relative to the repository root) ----

# Training data (in data/): 356 rows, min-max scaled to [0, 1].
# The metaheuristics and the cross-validation used this same file.
TRAIN_FILE = 'train_normal_rtf.csv'

# The same 356 training rows on the raw scale (written by denormalize.py). Used only for checks.
RAW_TRAIN_FILE = 'train_rtf.csv'

# Test sets (in data/), genes on the RAW scale: they are scaled in section 3. The files are never changed.
TEST_SETS = {
    'test': 'test_rtf_v2_cleaned.csv',   # hold-out: the 30 real test patients (20 negative, 10 positive)
    'NKI33': 'NKI33.csv',                # 33 patients (19 negative, 14 positive); 8 are also training patients
}
# TEST_NAME = 'NKI33'                      # the test set to evaluate: 'test' or 'NKI33'
TEST_NAME = 'test'                       # the test set to evaluate: 'test' or 'NKI33'
TEST_FILE = TEST_SETS[TEST_NAME]

# The same test set with its genes scaled (written in section 3.2, in data/): same rows, columns and labels.
# 'test_rtf_v2_cleaned.csv' -> 'test_rtf_v2_cleaned_normal.csv', 'NKI33.csv' -> 'NKI33_normal.csv'
SCALED_TEST_FILE = TEST_FILE.replace('.csv', '_normal.csv')

# The training "ruler": raw data set whose column ranges scaled TRAIN_FILE (see denormalize.py).
# Every test set is scaled with these same ranges. Do not change it when you change the test set.
SCALING_REFERENCE = 'rtfDataSet.csv'

# Gene subsets (0-based) and classifier of each of the 15 schemes
STEP = 'R3_v2'
SUBSETS_FILE = f'experiment_results/{STEP}/best_results_step_{STEP}.csv'

# 70-gene model (worst case in the manuscript): RF with this number of trees
ALL_GENES_RF_TREES = 60

# ---- Results (overwritten on every run; named after the test set) ----
RESULTS_15_SCHEMES = f'experiment_results/{STEP}/{TEST_NAME}_results_15_schemes.csv'
RESULTS_70_GENES = f'experiment_results/{STEP}/{TEST_NAME}_results_70_genes.csv'
RESULTS_EXCEL = f'experiment_results/{STEP}/{TEST_NAME}_results.xlsx'

## 2. Functions

The evaluation functions read the labels `Y_train` and `Y_test` from the notebook (section 3).

### 2.1 Data

In [5]:
def generateSubBM(features, BM):
    n, _ = BM.shape
    subBM = np.zeros((n, len(features)), dtype=float)
    for idx, value in enumerate(features):
        subBM[:, idx] = BM[:, value].reshape(n,)
    return subBM


def processDataSet(data_name):
    data = pd.read_csv('data/' + data_name)
    labels = data['label'].to_numpy(dtype=np.intp)
    data = data.drop(['label'], axis= 1).to_numpy()
    return data, labels

### 2.2 Metrics

In [18]:
def getMetrics(y_test, y_predict, y_probability):
    tn, fp, fn, tp = metrics.confusion_matrix(y_test, y_predict).ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    mse = metrics.mean_squared_error(y_test, y_predict)
    #tn, fp, fn, tp
    auc, f1, mcc = metrics.roc_auc_score(
        y_test,  y_probability[:, 1]), metrics.f1_score(y_test, y_predict), metrics.matthews_corrcoef(y_test, y_predict)
    
    dict_metrics = {
        'acc': accuracy,
        'mse': mse,
        'pre': precision,
        'rec': recall,
        'auc': auc,
        'f1': f1,
        'mcc': mcc,
        'spe': specificity,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp,
    }
    return dict_metrics

### 2.3 Classifiers

In [19]:
def runRandomForestClassifier(X_train, Y_train, X_test, Y_test, n_estimators):
    clf = RandomForestClassifier(n_estimators, random_state=45)
    clf.fit(X_train, Y_train)
    y_predict = clf.predict(X_test)
    y_probability = clf.predict_proba(X_test)
    dict_metrics = getMetrics(Y_test, y_predict, y_probability)
    return dict_metrics


def runLDAClassifier(X_train, Y_train, X_test, Y_test):
    clf = LinearDiscriminantAnalysis()
    clf.fit(X_train, Y_train)
    y_predict = clf.predict(X_test)
    y_probability = clf.predict_proba(X_test)
    dict_metrics = getMetrics(Y_test, y_predict, y_probability)
    return dict_metrics


def runKNNClassifier(X_train, Y_train, X_test, Y_test, k_neighbors):
    knn = KNeighborsClassifier(k_neighbors)
    knn.fit(X_train, Y_train)
    y_predict = knn.predict(X_test)
    y_probability = knn.predict_proba(X_test)
    dict_metrics = getMetrics(Y_test, y_predict, y_probability)
    return dict_metrics


def runQDAClassifier(X_train, Y_train, X_test, Y_test):
    clf = QuadraticDiscriminantAnalysis()
    clf.fit(X_train, Y_train)
    y_predict = clf.predict(X_test)
    y_probability = clf.predict_proba(X_test)
    dict_metrics = getMetrics(Y_test, y_predict, y_probability)
    return dict_metrics


def runSVMClassifier(X_train, Y_train, X_test, Y_test):
    clf = SVC(kernel='poly', probability=True)
    clf.fit(X_train, Y_train)
    y_probability = clf.predict_proba(X_test)
    y_predict = clf.predict(X_test)
    dict_metrics = getMetrics(Y_test, y_predict, y_probability)
    return dict_metrics


def runNaiveBayesClassifier(X_train, Y_train, X_test, Y_test):
    gnb = GaussianNB()
    gnb.fit(X_train, Y_train)
    y_predict = gnb.predict(X_test)
    y_probability = gnb.predict_proba(X_test) 
    dict_metrics = getMetrics(Y_test, y_predict, y_probability)
    return dict_metrics

### 2.4 Evaluation of one scheme

In [20]:
RESULTS_HEADER = 'name,classifier,auc,acc,pre,mse,rec,f1,mcc,spe,tn,fp,fn,tp\n'


def writeClassifier(resume_path, name_exp,  metrics_values, classifier_name):
    file = open(resume_path, 'a')
    file.write(f"{name_exp},{classifier_name},{metrics_values['auc']},{metrics_values['acc']},{metrics_values['pre']},{metrics_values['mse']},{metrics_values['rec']},{metrics_values['f1']},{metrics_values['mcc']},{metrics_values.get('spe','')},{metrics_values.get('tn','')},{metrics_values.get('fp','')},{metrics_values.get('fn','')},{metrics_values.get('tp','')}\n")
    file.close()


def evaluation_function_rf(name_exp, subset, X_train, X_test, n_estimators, resume_path):
    X_structured_train = generateSubBM(subset, X_train)
    X_structured_test = generateSubBM(subset, X_test)

    metrics_values = runRandomForestClassifier(X_structured_train, Y_train, X_structured_test, Y_test, n_estimators)
    writeClassifier(resume_path, name_exp, metrics_values, f'RF estimators: {n_estimators}')


def evaluation_function_lda(name_exp, subset, X_train, X_test, resume_path):
    X_structured_train = generateSubBM(subset, X_train)
    X_structured_test = generateSubBM(subset, X_test)

    metrics_values = runLDAClassifier(X_structured_train, Y_train, X_structured_test, Y_test)
    writeClassifier(resume_path, name_exp, metrics_values, 'LDA')


def evaluation_function_knn(name_exp, subset, X_train, X_test, k_neighbors, resume_path):
    X_structured_train = generateSubBM(subset, X_train)
    X_structured_test = generateSubBM(subset, X_test)

    metrics_values = runKNNClassifier(X_structured_train, Y_train, X_structured_test, Y_test, k_neighbors)
    writeClassifier(resume_path, name_exp, metrics_values, f'Knn k = {k_neighbors}')


def evaluation_function_qda(name_exp, subset, X_train, X_test, resume_path):
    X_structured_train = generateSubBM(subset, X_train)
    X_structured_test = generateSubBM(subset, X_test)

    metrics_values = runQDAClassifier(X_structured_train, Y_train, X_structured_test, Y_test)
    writeClassifier(resume_path, name_exp, metrics_values, f'QDA')


def evaluation_function_svm(name_exp, subset, X_train, X_test, resume_path):
    X_structured_train = generateSubBM(subset, X_train)
    X_structured_test = generateSubBM(subset, X_test)
    metrics_values = runSVMClassifier(X_structured_train, Y_train, X_structured_test, Y_test)
    writeClassifier(resume_path, name_exp, metrics_values, f'SVM Kernel = poly')


def evaluation_function_naive_bayes(name_exp, subset, X_train, X_test, resume_path):
    X_structured_train = generateSubBM(subset, X_train)
    X_structured_test = generateSubBM(subset, X_test)
    metrics_values = runNaiveBayesClassifier(X_structured_train, Y_train, X_structured_test, Y_test)
    writeClassifier(resume_path, name_exp, metrics_values, f'Naive bayes')

### 2.5 Evaluation of all schemes

In [21]:
def test_classifier_evaluation(subsets_file, output_file):
    """Evaluates every scheme of subsets_file on the test set and writes one row per scheme to output_file."""
    df = pd.read_csv(subsets_file)
    with open(output_file, 'w') as f:
        f.write(RESULTS_HEADER)

    for _, row in df.iterrows():
        name = row['Meta'] + '_' + row['OF']
        subset = ast.literal_eval(row['Subset'])
        classifier = row['Classifier']
        if classifier.startswith('Knn'):
            k_neighbors = int(classifier[len("Knn k = "):])
            print(f'>> {name}: KNN with k = {k_neighbors}')
            evaluation_function_knn(name, subset, X_train, X_test, k_neighbors, output_file)
        elif classifier.startswith('RF'):
            n_estimators = int(classifier[len("RF estimators: "):])
            print(f'>> {name}: RF with n = {n_estimators}')
            evaluation_function_rf(name, subset, X_train, X_test, n_estimators, output_file)
        elif classifier.startswith('LDA'):
            print(f'>> {name}: LDA')
            evaluation_function_lda(name, subset, X_train, X_test, output_file)
        elif classifier.startswith('QDA'):
            print(f'>> {name}: QDA')
            evaluation_function_qda(name, subset, X_train, X_test, output_file)
        elif classifier.startswith('SVM'):
            print(f'>> {name}: SVM')
            evaluation_function_svm(name, subset, X_train, X_test, output_file)
        elif classifier.startswith('Naive') or classifier.startswith('NB'):
            print(f'>> {name}: Naive bayes')
            evaluation_function_naive_bayes(name, subset, X_train, X_test, output_file)
        else:
            raise ValueError(f'{name}: unknown classifier {classifier!r}')

## 3. Dataset

### 3.1 Training data

In [6]:
X_train, Y_train = processDataSet(TRAIN_FILE)
print('TRAIN:', TRAIN_FILE, '| X:', X_train.shape, '| Y:', Y_train.shape)

TRAIN: train_normal_rtf.csv | X: (356, 70) | Y: (356,)


### 3.2 Test data

The test file must have exactly the training columns (genes 0-69 and label, no ID column) and be on the raw
scale. Its genes are scaled with the column ranges of `SCALING_REFERENCE`, the same ranges that scaled the
training data. The cell first checks that `SCALING_REFERENCE` really reproduces the training data.

In [7]:
train_columns = pd.read_csv(f'data/{TRAIN_FILE}', nrows=0).columns.tolist()
test_columns = pd.read_csv(f'data/{TEST_FILE}', nrows=0).columns.tolist()
assert test_columns == train_columns, (
    f'{TEST_FILE} must have the training columns (genes 0-69 and label) in the same order. '
    f'Extra: {sorted(set(test_columns) - set(train_columns))}, missing: {sorted(set(train_columns) - set(test_columns))}')

# SCALING_REFERENCE must be the training ruler: scaling the raw training rows with it must give TRAIN_FILE
raw_data, _ = processDataSet(SCALING_REFERENCE)
data_min, data_max = raw_data.min(axis=0), raw_data.max(axis=0)
raw_train, _ = processDataSet(RAW_TRAIN_FILE)
assert np.abs((raw_train - data_min) / (data_max - data_min) - X_train).max() < 1e-9, (
    f'{SCALING_REFERENCE} does not reproduce the scaling of {TRAIN_FILE}: keep SCALING_REFERENCE = rtfDataSet.csv')

X_test, Y_test = processDataSet(TEST_FILE)
assert X_test.min() < 0, f'{TEST_FILE} looks already scaled: this cell expects the raw scale'
X_test = (X_test - data_min) / (data_max - data_min)
print('TEST:', TEST_NAME, '=', TEST_FILE, '| X:', X_test.shape, '| Y:', Y_test.shape)

TEST: test = test_rtf_v2_cleaned.csv | X: (30, 70) | Y: (30,)


In [8]:
# Write the test set with its genes scaled: the original file, same rows in the same order and same labels,
# with only the gene columns replaced by X_test. The raw test file is not changed.
assert SCALED_TEST_FILE != TEST_FILE, 'SCALED_TEST_FILE would overwrite the raw test file'
gene_columns = [c for c in train_columns if c != 'label']
test_scaled = pd.read_csv(f'data/{TEST_FILE}')
test_scaled[gene_columns] = X_test
assert (test_scaled['label'].to_numpy() == Y_test).all()
test_scaled.to_csv(f'data/{SCALED_TEST_FILE}', index=False)
print('Written to', f'data/{SCALED_TEST_FILE}', test_scaled.shape)

Written to data/test_rtf_v2_cleaned_normal.csv (30, 71)


### 3.3 Checks and summary

In [24]:
assert X_test.shape[1] == X_train.shape[1], 'train and test must have the same 70 genes'
tolerance = 1e-9   # the scaled files hold values such as 1.0000000000000002
assert -tolerance <= X_train.min() and X_train.max() <= 1 + tolerance, 'training data must be scaled to [0, 1]'

# Patients outside the range of SCALING_REFERENCE get values below 0 or above 1. This is expected for patients
# that are not in rtfDataSet.csv (most of NKI33). The values are kept as they are.
outside = (X_test < -tolerance) | (X_test > 1 + tolerance)
print(f'Test values outside [0, 1]: {outside.sum()} of {X_test.size}, in {outside.any(axis=1).sum()} patients')

# Test patients that are also training patients: the models have seen them, so their results are optimistic.
distance = np.abs(X_test[:, None, :] - X_train[None, :, :]).max(axis=2)
in_training = np.flatnonzero(distance.min(axis=1) < 1e-9)
if len(in_training):
    print(f'WARNING: {len(in_training)} test patients are also in the training data '
          f'(rows {in_training.tolist()} of {TEST_FILE}). They are kept in the evaluation.')

pd.DataFrame({
    'file': [TRAIN_FILE, TEST_FILE],
    'patients': [len(Y_train), len(Y_test)],
    'genes': [X_train.shape[1], X_test.shape[1]],
    'negative (0)': [int((Y_train == 0).sum()), int((Y_test == 0).sum())],
    'positive (1)': [int((Y_train == 1).sum()), int((Y_test == 1).sum())],
    'min': [X_train.min(), X_test.min()],
    'max': [X_train.max(), X_test.max()],
    'values outside [0, 1]': [0, int(outside.sum())],
    'patients also in training': ['-', len(in_training)],
}, index=['train', 'test']).round(3)

Test values outside [0, 1]: 0 of 2100, in 0 patients


,file,patients,genes,negative (0),positive (1),min,max,"values outside [0, 1]",patients also in training
train,train_normal_rtf.csv,356,70,174,182,0.0,1.0,0,-
test,test_rtf_v2_cleaned.csv,30,70,20,10,0.0,1.0,0,0


## 4. Evaluation

### 4.1 The 15 selected schemes

In [25]:
test_classifier_evaluation(SUBSETS_FILE, RESULTS_15_SCHEMES)
print('Written to', RESULTS_15_SCHEMES)

>> DE_chi2: RF with n = 100
>> DE_gain_ratio: RF with n = 60
>> DE_mi: Naive bayes
>> DE_relieff: KNN with k = 3
>> DE_sim_uncertainty: RF with n = 60
>> GA_chi2: SVM


/mnt/Cris-SW/ResearchV2/ASP-ML/Metaheuristics-Mammaprint/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


>> GA_gain_ratio: RF with n = 80
>> GA_mi: RF with n = 60
>> GA_relieff: KNN with k = 3
>> GA_sim_uncertainty: RF with n = 80
>> SA_chi2: RF with n = 100
>> SA_gain_ratio: RF with n = 80
>> SA_mi: RF with n = 100
>> SA_relieff: KNN with k = 5
>> SA_sim_uncertainty: RF with n = 60
Written to experiment_results/R3_v2/test_results_15_schemes.csv


### 4.2 The 70-gene model (all genes)

In [26]:
all_genes = list(range(X_train.shape[1]))
with open(RESULTS_70_GENES, 'w') as f:
    f.write(RESULTS_HEADER)
evaluation_function_rf('ALL_FEATURES', all_genes, X_train, X_test, ALL_GENES_RF_TREES, RESULTS_70_GENES)
print('Written to', RESULTS_70_GENES)

Written to experiment_results/R3_v2/test_results_70_genes.csv


## 5. Results

`rec` is Sn and `spe` is Sp. Values are rounded here for reading only; the files keep full precision.

In [27]:
columns = ['name', 'classifier', 'f1', 'pre', 'rec', 'spe', 'acc', 'auc', 'tn', 'fp', 'fn', 'tp']
results = pd.concat([pd.read_csv(RESULTS_15_SCHEMES), pd.read_csv(RESULTS_70_GENES)], ignore_index=True)
results[columns].round(2)

,name,classifier,f1,pre,rec,spe,acc,auc,tn,fp,fn,tp
0,DE_chi2,RF estimators: 100,0.69,0.56,0.9,0.65,0.73,0.76,13,7,1,9
1,DE_gain_ratio,RF estimators: 60,0.74,0.59,1.0,0.65,0.77,0.88,13,7,0,10
2,DE_mi,Naive bayes,0.58,0.43,0.9,0.40,0.57,0.68,8,12,1,9
3,DE_relieff,Knn k = 3,0.72,0.60,0.9,0.70,0.77,0.94,14,6,1,9
4,DE_sim_uncertainty,RF estimators: 60,0.87,0.77,1.0,0.85,0.90,0.93,17,3,0,10
5,GA_chi2,SVM Kernel = poly,0.54,0.44,0.7,0.55,0.60,0.56,11,9,3,7
6,GA_gain_ratio,RF estimators: 80,0.52,0.46,0.6,0.65,0.63,0.68,13,7,4,6
7,GA_mi,RF estimators: 60,0.48,0.45,0.5,0.70,0.63,0.66,14,6,5,5
8,GA_relieff,Knn k = 3,0.77,0.62,1.0,0.70,0.80,0.90,14,6,0,10
9,GA_sim_uncertainty,RF estimators: 80,0.67,0.57,0.8,0.70,0.73,0.69,14,6,2,8


In [28]:
OF_LABELS = {'mi': 'aMI', 'sim_uncertainty': 'aSU', 'gain_ratio': 'aGR', 'relieff': 'aReliefF', 'chi2': 'aCHI2'}

table = results.copy()          # results stays unchanged, so this cell can be run again
table[['Meta', 'OF']] = table['name'].str.split('_', n=1, expand=True)
table = table.drop(columns=['name'])
table['OF'] = table['OF'].map(OF_LABELS).fillna('-')
table = table.rename(columns={'name': 'Scheme', 'classifier': 'Classifier', 'f1': 'F1-score', 'pre': 'Precision',
                              'rec': 'Recall', 'spe': 'Specificity', 'acc': 'Accuracy', 'auc': 'AUC', 'tn': 'TN', 'fp': 'FP', 'fn': 'FN', 'tp': 'TP'})
first = ['Meta', 'OF', 'Classifier']
table = table[first + [c for c in table.columns if c not in first]]
table.to_excel(RESULTS_EXCEL, index=False)
print('Written to', RESULTS_EXCEL)

Written to experiment_results/R3_v2/test_results.xlsx
